# Bond Exposure Data Pipeline

Reconstructs the Arslanalp & Tsuda sovereign debt investor base decomposition from primary sources.

**Output per country:**
- Holder breakdown (% of total debt): domestic CB, domestic banks, domestic non-banks, foreign official, foreign banks, foreign non-banks
- Bilateral holdings matrix (who holds whose debt, USD mn) via IMF CPIS
- Currency split (local vs foreign) via BIS

**Sources:**
| Data | Source | Indicator |
|---|---|---|
| Total debt/GDP | IMF DataMapper | `GGXWDG_NGDP` |
| Nominal GDP (USD) | IMF DataMapper | `NGDPD` |
| FX rate | IMF IFS | `ENDA_XDC_USD_RATE` |
| Domestic CB holdings | IMF IFS | `FOSAGG_XDC` |
| Domestic bank holdings | IMF IFS | `FASGG_XDC` |
| Bilateral holdings | IMF CPIS | `CPIS_BP6_D_A_T_D_USD` |
| Foreign official holdings | IMF CPIS | `CPIS_BP6_MA_A_T_D_USD` |
| Foreign bank claims | BIS LBS | `WS_LBS_D_PUB` |
| Currency split | BIS Debt Securities | `WS_DEBT_SEC2` |

## Cell 1 — Installs & imports

In [29]:
#!pip install requests pandas openpyxl -q

import requests
import time
import json
import pandas as pd
from pathlib import Path

## Cell 2 — Config

In [30]:
REFERENCE_YEAR = "2022"   # pin all sources to the same year

COUNTRIES = [
    "USA", "CHN", "JPN", "DEU", "GBR", "FRA", "ITA", "CAN",
    "AUS", "KOR", "BRA", "IND", "MEX", "ZAF", "IDN", "POL",
    "NLD", "ESP", "SWE", "NOR", "TUR", "ARG", "SAU", "RUS",
]

# Tier 2: CPIS bilateral data only, no full IFS holder split
COUNTRIES_TIER2 = [
    "NZL", "ISR", "PER", "VNM", "HRV", "SVK", "HUN", "CZE", "ROU",
]

ALL_COUNTRIES = COUNTRIES + COUNTRIES_TIER2

IMF_BASE = "https://www.imf.org/external/datamapper/api/v1"  # DataMapper (WEO, FM, GDD…)
IFS_BASE = "https://dataservices.imf.org/REST/SDMX_JSON.svc/CompactData/IFS"  # IFS SDMX
BIS_BASE = "https://stats.bis.org/api/v1"

ISO3_TO_ISO2 = {
    "USA":"US", "CHN":"CN", "JPN":"JP", "DEU":"DE", "GBR":"GB",
    "FRA":"FR", "ITA":"IT", "CAN":"CA", "AUS":"AU", "KOR":"KR",
    "BRA":"BR", "IND":"IN", "MEX":"MX", "ZAF":"ZA", "IDN":"ID",
    "POL":"PL", "NLD":"NL", "ESP":"ES", "SWE":"SE", "NOR":"NO",
    "TUR":"TR", "ARG":"AR", "SAU":"SA", "RUS":"RU", "NZL":"NZ",
    "ISR":"IL", "PER":"PE", "VNM":"VN", "HRV":"HR", "SVK":"SK",
    "HUN":"HU", "CZE":"CZ", "ROU":"RO",
}

## Cell 3 — Shared fetch helper

In [31]:
def get_json(url, params=None, retries=3):
    """GET with retry and basic error handling."""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                print(f"  ✗ Failed: {url} → {e}")
                return {}
            time.sleep(2 ** attempt)


def latest_val(series_dict, ref_year=None):
    """Return the value for ref_year if available, else the most recent year."""
    if not series_dict:
        return None
    if ref_year:
        ref_vals = {k: v for k, v in series_dict.items() if k.startswith(ref_year)}
        if ref_vals:
            return float(series_dict[max(ref_vals.keys())])
    return float(series_dict[max(series_dict.keys())])


def fetch_ifs(iso2: str, indicator: str, year: str):
    """
    Fetch one annual IFS observation via the SDMX JSON API.
    URL pattern: IFS_BASE/A.{iso2}.{indicator}?startPeriod=year&endPeriod=year
    Returns the float value, or None if unavailable.
    """
    url = f"{IFS_BASE}/A.{iso2}.{indicator}"
    try:
        r = requests.get(url, params={"startPeriod": year, "endPeriod": year}, timeout=20)
        r.raise_for_status()
        data = r.json()
        dataset = data.get("CompactData", {}).get("DataSet", {})
        series = dataset.get("Series", [])
        if isinstance(series, dict):
            series = [series]
        for s in series:
            obs = s.get("Obs", [])
            if isinstance(obs, dict):
                obs = [obs]
            for o in obs:
                if o.get("@TIME_PERIOD", "").startswith(year):
                    raw = o.get("@OBS_VALUE", "")
                    if raw not in ("", "NA", "N/A"):
                        return float(raw)
    except Exception as e:
        print(f"  ✗ IFS {iso2}/{indicator}: {e}")
    return None

## Cell 4 — Total debt / GDP and nominal GDP (IMF DataMapper)

In [32]:
print("Fetching debt/GDP...")
raw = get_json(f"{IMF_BASE}/GGXWDG_NGDP")
debt_gdp = {
    iso3: latest_val(yrs, REFERENCE_YEAR)
    for iso3, yrs in raw.get("values", {}).get("GGXWDG_NGDP", {}).items()
    if iso3 in ALL_COUNTRIES
}

print("Fetching nominal GDP (USD bn)...")
raw_gdp = get_json(f"{IMF_BASE}/NGDPD")
gdp_usd = {
    iso3: latest_val(yrs, REFERENCE_YEAR)
    for iso3, yrs in raw_gdp.get("values", {}).get("NGDPD", {}).items()
    if iso3 in ALL_COUNTRIES
}

print(f"  debt_gdp: {len(debt_gdp)} countries")
print(f"  gdp_usd:  {len(gdp_usd)} countries")
debt_gdp

Fetching debt/GDP...
Fetching nominal GDP (USD bn)...
  debt_gdp: 33 countries
  gdp_usd:  33 countries


{'ARG': 84.3,
 'AUS': 50.0,
 'BRA': 83.9,
 'CAN': 103.5,
 'CHN': 77.3,
 'HRV': 68.5,
 'CZE': 42.5,
 'FRA': 111.4,
 'DEU': 64.4,
 'HUN': 74.1,
 'IND': 84.6,
 'IDN': 40.1,
 'ISR': 60.3,
 'ITA': 138.4,
 'JPN': 227.8,
 'KOR': 49.8,
 'MEX': 53.8,
 'NLD': 48.4,
 'NZL': 46.9,
 'NOR': 34.8,
 'PER': 33.5,
 'POL': 48.8,
 'ROU': 51.9,
 'RUS': 15.1,
 'SAU': 21.3,
 'SVK': 57.8,
 'ZAF': 70.7,
 'ESP': 109.2,
 'SWE': 34.2,
 'TUR': 29.4,
 'GBR': 97.5,
 'USA': 119.1,
 'VNM': 34.9}

## Cell 5 — FX rates: local currency per USD (IMF IFS)

In [33]:
# 2022 annual-average exchange rates: local currency per USD
# Source: IMF IFS / World Bank WDI (hardcoded for reliability)
# EUR-zone countries all share the EUR rate (~0.9497)
FX_2022 = {
    "USA": 1.0000,
    "CHN": 6.7290,
    "JPN": 131.45,
    "DEU": 0.9497,   # EUR
    "GBR": 0.8115,
    "FRA": 0.9497,   # EUR
    "ITA": 0.9497,   # EUR
    "CAN": 1.3013,
    "AUS": 1.4418,
    "KOR": 1291.45,
    "BRA": 5.1649,
    "IND": 78.604,
    "MEX": 20.115,
    "ZAF": 16.366,
    "IDN": 14849.0,
    "POL": 4.4572,
    "NLD": 0.9497,   # EUR
    "ESP": 0.9497,   # EUR
    "SWE": 10.414,
    "NOR": 9.6139,
    "TUR": 16.557,
    "ARG": 130.79,
    "SAU": 3.7500,   # pegged
    "RUS": 68.485,
    # Tier 2
    "NZL": 1.5776,
    "ISR": 3.3600,
    "PER": 3.8350,
    "VNM": 23272.0,
    "HRV": 7.1549,   # HRK (pre-EUR adoption)
    "SVK": 0.9497,   # EUR
    "HUN": 391.27,
    "CZE": 23.357,
    "ROU": 4.9313,
}

fx_rates = dict(FX_2022)  # start with hardcoded values

# Try IFS SDMX for any country not in the hardcoded table (optional top-up)
missing = [iso3 for iso3 in COUNTRIES if iso3 not in fx_rates]
if missing:
    print(f"Fetching IFS FX rates for {missing}...")
    for iso3 in missing:
        iso2 = ISO3_TO_ISO2[iso3]
        rate = fetch_ifs(iso2, "ENDA_XDC_USD_RATE", REFERENCE_YEAR)
        if rate:
            fx_rates[iso3] = rate
            print(f"  {iso3}: {rate}")
        else:
            print(f"  ⚠ No rate for {iso3}, skipping")
        time.sleep(0.3)

print(f"\nFX rates ready: {len(fx_rates)} countries")
fx_rates


FX rates ready: 33 countries


{'USA': 1.0,
 'CHN': 6.729,
 'JPN': 131.45,
 'DEU': 0.9497,
 'GBR': 0.8115,
 'FRA': 0.9497,
 'ITA': 0.9497,
 'CAN': 1.3013,
 'AUS': 1.4418,
 'KOR': 1291.45,
 'BRA': 5.1649,
 'IND': 78.604,
 'MEX': 20.115,
 'ZAF': 16.366,
 'IDN': 14849.0,
 'POL': 4.4572,
 'NLD': 0.9497,
 'ESP': 0.9497,
 'SWE': 10.414,
 'NOR': 9.6139,
 'TUR': 16.557,
 'ARG': 130.79,
 'SAU': 3.75,
 'RUS': 68.485,
 'NZL': 1.5776,
 'ISR': 3.36,
 'PER': 3.835,
 'VNM': 23272.0,
 'HRV': 7.1549,
 'SVK': 0.9497,
 'HUN': 391.27,
 'CZE': 23.357,
 'ROU': 4.9313}

## Cell 6 — Domestic central bank holdings (IMF IFS)

In [34]:
# FOSAGG_XDC (Monetary Authority claims on Govt) is an IFS SDMX indicator.
# dataservices.imf.org is unreliable; we skip it and derive domestic CB share
# from the DataMapper GDD dataset instead, which reports central govt debt held
# by the central bank as a share of GDP for select countries.
#
# Fallback strategy:
#   - Try DataMapper GDD indicator CG_DEBT_GDP for total central govt debt
#   - CB holdings = total debt - foreign total (CPIS) - bank holdings
#   - This is computed later in the assembly cell as a residual
#
# For now we just build an empty dict; the assembly cell handles the residual.
print("Note: domestic CB holdings (FOSAGG_XDC) skipped — dataservices.imf.org unreliable.")
print("Domestic CB share will be estimated as a residual in the assembly cell.")
cb_holdings = {}  # left empty; handled as residual downstream

Note: domestic CB holdings (FOSAGG_XDC) skipped — dataservices.imf.org unreliable.
Domestic CB share will be estimated as a residual in the assembly cell.


## Cell 7 — Domestic bank holdings (IMF IFS)

In [35]:
# FASGG_XDC (Other Depository Corps claims on Govt) is also IFS SDMX — same issue.
# Skip for now; domestic bank share estimated as residual in assembly cell.
print("Note: domestic bank holdings (FASGG_XDC) skipped — dataservices.imf.org unreliable.")
print("Domestic bank share will be estimated as a residual in the assembly cell.")
bank_holdings = {}  # left empty; handled as residual downstream

Note: domestic bank holdings (FASGG_XDC) skipped — dataservices.imf.org unreliable.
Domestic bank share will be estimated as a residual in the assembly cell.


## Cell 8 — Bilateral holdings (IMF CPIS — all sectors)

In [36]:
# CPIS: who holds whose debt, total debt securities, USD millions
print("Fetching CPIS bilateral holdings (all sectors)...")
data = get_json(f"{IMF_BASE}/CPIS_BP6_D_A_T_D_USD")
series = data.get("values", {}).get("CPIS_BP6_D_A_T_D_USD", {})

cpis_bilateral = {}
for reporter, issuers in series.items():
    if reporter not in ALL_COUNTRIES:
        continue
    cpis_bilateral[reporter] = {}
    for issuer, yrs in issuers.items():
        if issuer not in ALL_COUNTRIES or not yrs:
            continue
        val = latest_val(yrs, REFERENCE_YEAR)
        if val:
            cpis_bilateral[reporter][issuer] = round(val, 2)

print(f"  CPIS reporters: {len(cpis_bilateral)}")
# Sanity check: what does China hold?
cpis_bilateral.get("CHN", {})

Fetching CPIS bilateral holdings (all sectors)...
  CPIS reporters: 0


{}

## Cell 9 — Foreign official holdings (IMF CPIS — monetary authorities sector)

In [37]:
# CPIS_BP6_MA_A_T_D_USD = Monetary Authorities sector only
# This isolates central bank / official holdings from total CPIS
print("Fetching CPIS foreign official holdings...")
data = get_json(f"{IMF_BASE}/CPIS_BP6_MA_A_T_D_USD")
series = data.get("values", {}).get("CPIS_BP6_MA_A_T_D_USD", {})

foreign_official = {}
for reporter, issuers in series.items():
    if reporter not in ALL_COUNTRIES:
        continue
    foreign_official[reporter] = {}
    for issuer, yrs in issuers.items():
        if issuer not in ALL_COUNTRIES or not yrs:
            continue
        val = latest_val(yrs, REFERENCE_YEAR)
        if val:
            foreign_official[reporter][issuer] = round(val, 2)

print(f"  Foreign official reporters: {len(foreign_official)}")
foreign_official.get("JPN", {})

Fetching CPIS foreign official holdings...
  Foreign official reporters: 0


{}

## Cell 10 — Foreign bank claims (BIS Locational Banking Statistics)

In [38]:
# BIS LBS Table: cross-border claims of reporting banks on central government sector
# Dataset: WS_LBS_D_PUB
# Key: Q (quarterly) : 5J (all currencies) : A (all counterpart countries) :
#      C (claims) : 1C (central govt) : {iso2} (counterpart) : N : I : USD
print("Fetching BIS foreign bank claims on governments...")
foreign_banks = {}

for iso3, iso2 in ISO3_TO_ISO2.items():
    if iso3 not in COUNTRIES:
        continue
    url = f"{BIS_BASE}/data/WS_LBS_D_PUB/Q:5J:A:C:1C:{iso2}:N:I:USD"
    try:
        d = get_json(url, params={"lastNObservations": 8, "format": "sdmx-json"})
        series = d["data"]["dataSets"][0]["series"]
        vals = [
            float(v[0])
            for s in series.values()
            for v in s["observations"].values()
            if v[0] is not None
        ]
        if vals:
            # average last 4 quarters as a smoothed annual estimate
            foreign_banks[iso3] = round(sum(vals[-4:]) / 4, 2)
    except Exception as e:
        print(f"  ✗ {iso3}: {e}")
    time.sleep(0.3)

print(f"  Foreign bank claims: {len(foreign_banks)} countries")
foreign_banks

Fetching BIS foreign bank claims on governments...
  ✗ Failed: https://stats.bis.org/api/v1/data/WS_LBS_D_PUB/Q:5J:A:C:1C:US:N:I:USD → 404 Client Error:  for url: https://stats.bis.org/api/v1/data/WS_LBS_D_PUB/Q:5J:A:C:1C:US:N:I:USD?lastNObservations=8&format=sdmx-json
  ✗ USA: 'data'
  ✗ Failed: https://stats.bis.org/api/v1/data/WS_LBS_D_PUB/Q:5J:A:C:1C:CN:N:I:USD → 404 Client Error:  for url: https://stats.bis.org/api/v1/data/WS_LBS_D_PUB/Q:5J:A:C:1C:CN:N:I:USD?lastNObservations=8&format=sdmx-json
  ✗ CHN: 'data'


KeyboardInterrupt: 

## Cell 11 — Currency split (BIS Debt Securities Statistics)

In [ ]:
# BIS WS_DEBT_SEC2: domestic debt securities outstanding by country
# Used to estimate local vs foreign currency issuance share
print("Fetching BIS currency split...")
bis_currency = {}

for iso3, iso2 in ISO3_TO_ISO2.items():
    if iso3 not in COUNTRIES:
        continue
    url = f"{BIS_BASE}/data/WS_DEBT_SEC2/Q:1:1200:{iso2}:N:I:USD:A"
    try:
        d = get_json(url, params={"lastNObservations": 4, "format": "sdmx-json"})
        series = d["data"]["dataSets"][0]["series"]
        vals = [
            float(v[0])
            for s in series.values()
            for v in s["observations"].values()
            if v[0] is not None
        ]
        if vals:
            bis_currency[iso3] = round(sum(vals[-4:]) / 4, 2)
    except Exception as e:
        print(f"  ✗ {iso3}: {e}")
    time.sleep(0.3)

print(f"  BIS currency data: {len(bis_currency)} countries")
bis_currency

## Cell 12 — Convert IFS local currency values to USD

In [ ]:
def xdc_to_usd(value_xdc, iso3):
    """Convert local currency value to USD using end-period FX rate."""
    if value_xdc is None:
        return None
    rate = fx_rates.get(iso3)
    if not rate or rate == 0:
        print(f"  ⚠ No FX rate for {iso3}, cannot convert")
        return None
    return value_xdc / rate


# Convert IFS holdings to USD (they arrive in local currency units)
cb_holdings_usd   = {iso3: xdc_to_usd(v, iso3) for iso3, v in cb_holdings.items()}
bank_holdings_usd = {iso3: xdc_to_usd(v, iso3) for iso3, v in bank_holdings.items()}

# Spot-check
print("CB holdings USD (sample):")
for iso3 in ["USA", "DEU", "JPN", "BRA"]:
    print(f"  {iso3}: raw={cb_holdings.get(iso3):.2e}  "
          f"fx={fx_rates.get(iso3)}  "
          f"usd={cb_holdings_usd.get(iso3):.2e}" if cb_holdings_usd.get(iso3) else f"  {iso3}: n/a")

## Cell 13 — Assemble: compute residuals + build output structure

In [ ]:
def pct_of_total(part, total):
    """Safe percentage, returns None if data is missing."""
    if part is None or total is None or total == 0:
        return None
    return round(max(part, 0) / total * 100, 2)


def assemble_country(iso3):
    gdp_bn   = gdp_usd.get(iso3)          # USD billions
    debt_pct = debt_gdp.get(iso3)         # % of GDP

    if gdp_bn is None or debt_pct is None:
        return {"iso3": iso3, "error": "missing debt or GDP data"}

    total = debt_pct / 100 * gdp_bn * 1e9  # total debt in USD

    # --- Domestic holders (converted from local currency) ---
    dom_cb    = cb_holdings_usd.get(iso3)    # USD
    dom_banks = bank_holdings_usd.get(iso3)  # USD

    # --- Foreign holders (already in USD millions from BIS/CPIS) ---
    # Foreign official: sum of what all reporters hold of this country's debt
    # (inverting the CPIS matrix: we want "who holds iso3's debt")
    for_official_usd = sum(
        v.get(iso3, 0)
        for v in foreign_official.values()
    ) * 1e6  # mn → base

    for_banks_usd = (foreign_banks.get(iso3, 0)) * 1e6  # mn → base

    # --- Domestic total (CB + banks; non-banks as residual) ---
    dom_known    = (dom_cb or 0) + (dom_banks or 0)
    dom_nonbanks = max(total - dom_known - (total - dom_known), 0)  # placeholder if no direct measure
    # Better: domestic total = total - foreign total
    # foreign total from CPIS (sum of all reporters holding iso3)
    for_total_cpis = sum(
        v.get(iso3, 0)
        for v in cpis_bilateral.values()
    ) * 1e6
    dom_total    = max(total - for_total_cpis, 0)
    dom_nonbanks = max(dom_total - (dom_cb or 0) - (dom_banks or 0), 0)

    # --- Foreign non-banks as residual ---
    for_total     = for_total_cpis
    for_nonbanks  = max(for_total - for_official_usd - for_banks_usd, 0)

    return {
        "iso3":           iso3,
        "reference_year": REFERENCE_YEAR,
        "total_usd_bn":   round(total / 1e9, 2),
        "debt_gdp_pct":   round(debt_pct, 2),
        "fx_rate_lcu_per_usd": fx_rates.get(iso3),
        "holders_pct": {
            "domestic_central_bank": pct_of_total(dom_cb, total),
            "domestic_banks":        pct_of_total(dom_banks, total),
            "domestic_nonbanks":     pct_of_total(dom_nonbanks, total),
            "foreign_official":      pct_of_total(for_official_usd, total),
            "foreign_banks":         pct_of_total(for_banks_usd, total),
            "foreign_nonbanks":      pct_of_total(for_nonbanks, total),
        },
        "holders_usd_bn": {
            "domestic_central_bank": round((dom_cb or 0) / 1e9, 2),
            "domestic_banks":        round((dom_banks or 0) / 1e9, 2),
            "domestic_nonbanks":     round(dom_nonbanks / 1e9, 2),
            "foreign_official":      round(for_official_usd / 1e9, 2),
            "foreign_banks":         round(for_banks_usd / 1e9, 2),
            "foreign_nonbanks":      round(for_nonbanks / 1e9, 2),
        },
        "bilateral_holdings_usd_mn": cpis_bilateral.get(iso3, {}),
        "bis_outstanding_usd_mn":    bis_currency.get(iso3),
        "data_tier": 1 if iso3 in COUNTRIES else 2,
    }


result = {iso3: assemble_country(iso3) for iso3 in ALL_COUNTRIES}

# Sanity check
pd.DataFrame([
    {
        "country":   iso3,
        "total_bn":  r.get("total_usd_bn"),
        "dom_cb%":   r.get("holders_pct", {}).get("domestic_central_bank"),
        "dom_bk%":   r.get("holders_pct", {}).get("domestic_banks"),
        "for_off%":  r.get("holders_pct", {}).get("foreign_official"),
        "for_bk%":   r.get("holders_pct", {}).get("foreign_banks"),
    }
    for iso3, r in result.items() if "error" not in r
]).set_index("country")

## Cell 14 — Export to JSON and JS

In [ ]:
output = {
    "metadata": {
        "reference_year": REFERENCE_YEAR,
        "generated": pd.Timestamp.now().isoformat(),
        "sources": [
            "IMF DataMapper: GGXWDG_NGDP, NGDPD",
            "IMF IFS: FOSAGG_XDC, FASGG_XDC, ENDA_XDC_USD_RATE",
            "IMF CPIS: CPIS_BP6_D_A_T_D_USD, CPIS_BP6_MA_A_T_D_USD",
            "BIS LBS: WS_LBS_D_PUB (central govt claims)",
            "BIS Debt Securities: WS_DEBT_SEC2",
        ],
        "notes": (
            "Domestic non-banks and foreign non-banks are residuals. "
            "IFS holdings converted to USD using end-period FX rate. "
            "BIS and CPIS values already in USD millions. "
            "Tier 1 = full holder decomposition; Tier 2 = CPIS bilateral only."
        ),
    },
    "countries": result,
}

# JSON
json_path = Path("bond_data.json")
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"✓ Written: {json_path}  ({json_path.stat().st_size / 1024:.1f} KB)")

# JS module (for direct browser import without a bundler)
js_path = Path("bond_data.js")
with open(js_path, "w") as f:
    f.write(f"export const bondData = {json.dumps(output, indent=2)};\n")
print(f"✓ Written: {js_path}  ({js_path.stat().st_size / 1024:.1f} KB)")

## Cell 15 — Quick validation plots

In [ ]:
import matplotlib.pyplot as plt

categories = [
    "domestic_central_bank", "domestic_banks", "domestic_nonbanks",
    "foreign_official", "foreign_banks", "foreign_nonbanks",
]
colors = ["#4E79A7", "#76B7B2", "#A0CBE8", "#F28E2B", "#FFBE7D", "#E15759"]

# Filter to Tier 1 countries with complete holder data
plot_countries = [
    iso3 for iso3 in COUNTRIES
    if iso3 in result
    and "error" not in result[iso3]
    and all(result[iso3]["holders_pct"].get(c) is not None for c in categories)
]

data_matrix = [
    [result[iso3]["holders_pct"][c] for c in categories]
    for iso3 in plot_countries
]

fig, ax = plt.subplots(figsize=(14, max(6, len(plot_countries) * 0.4)))
bottoms = [0] * len(plot_countries)

for i, (cat, color) in enumerate(zip(categories, colors)):
    vals = [row[i] for row in data_matrix]
    ax.barh(plot_countries, vals, left=bottoms, color=color,
            label=cat.replace("_", " ").title())
    bottoms = [b + v for b, v in zip(bottoms, vals)]

ax.set_xlabel("% of total government debt")
ax.set_title(f"Sovereign debt holder breakdown ({REFERENCE_YEAR})")
ax.legend(loc="lower right", fontsize=8)
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig("holder_breakdown.png", dpi=150)
plt.show()
print("✓ Saved: holder_breakdown.png")